<div>
<img src=https://www.institutedata.com/wp-content/uploads/2019/10/iod_h_tp_primary_c.svg width="300">
</div>

# Lab 3.2.2
# *Mining Q&A content on Stack Exchange*

[Stack Exchange](https://stackexchange.com/about) is a collection of question-and-answer websites where users have the ability to pose, answer and upvote or downvote questions and answers on a wide variety of topics. [Stack Overflow](https://stackoverflow.com) in particular is used for programming related topics.

Data from the sites can be obtained in semi-structured form via the [Stack Exchange API](https://api.stackexchange.com/docs).

The purpose of this lab is to retrieve and analyse real-world semi-structured data, while introducing API concepts and enhancing Python programming skills for processing data from a public API.

If you have not already done so, visit https://stackoverflow.com to explore the most popular site on the Stack Exchange network. Select a post and observe how a number of answers may be provided. Questions and answers can be commented on and can be upvoted/downvoted.

Questions additionally have tags. As an example, visit

https://stackoverflow.com/questions/tagged/python

to see questions with the "python" tag.

We can view the same content via the Stack Exchange API at:

https://api.stackexchange.com/2.3/questions?order=desc&sort=creation&tagged=python&site=stackoverflow

Note that the same data is present on both sites, but the latter is in JSON form for easier consumption by computers.

In this lab we use the StackAPI wrapper as a Pythonic interface to Stack Exchange API. Further documentation can be found at https://stackapi.readthedocs.io/en/latest/. The benefit of using StackAPI versus the requests library to access the URL is that it provides a cleaner interface and is able to handle pagination (retrieving data from multiple pages) more effectively.

### 1. Install the StackAPI library

In [ ]:
!pip install stackapi

### 2. Load Python Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML # to display HTML as markdown
from stackapi import StackAPI

### 3. Viewing the sites

In [ ]:
sites = StackAPI().fetch('sites')

Inspecting the `sites` object you will notice that the key `items` contains most of the data. `sites['items']` is a list of dictionaries, with each dictionary having a variety of information about each site.

In [ ]:
sites

**Exercise**: Print the site_urls of all items in the `sites` object. How many sites are there?

In [ ]:
# ANSWER - hint: print d['site_url'] for all d in sites['items']


In [ ]:
# ANSWER - number of items


### 4. Creating a site object
Let us focus on the Stack Overflow website in particular.

In [ ]:
SITE = StackAPI('stackoverflow')

Referring to https://api.stackexchange.com/docs, if the API documentation shows:

/2.3/path?param1=val1&param2=val2...

Then in StackAPI we would use
SITE.fetch('path')
with additional parameters listed as arguments to SITE.fetch().

For example, above we called
https://api.stackexchange.com/2.3/questions?order=desc&sort=creation&tagged=python&site=stackoverflow

Using StackAPI the same data can be fetched using the following code:

In [ ]:
data = SITE.fetch(
    'questions',
    order='desc',
    sort='votes',
    tagged='python'
)

In [ ]:
data

Note from this object that there is a `quota` of 300 API calls.

We can now view the highest scoring titles using the following for loop:

In [ ]:
for q in data["items"][:5]:
    print(f'{q["score"]} : {q["title"]}')

Further inputs to the fetch method are listed below, some of which will be used later. Curly braces ({ids} or {tags}) become keyword arguments:

- questions/{ids}, ids=[(list of question ids)]
- answers
- users
- users/{ids}, ids=[(list of user ids)]
- tags
- tags/{tags}/info', tags=[(list of tags)]
- search, intitle='(string found in title)'
- search/advanced
- privileges

**Exercise:** How much reputation is required to `Create new tags` in the site? (use SITE.fetch('privileges'))

In [ ]:
# ANSWER


In [ ]:
# ANSWER HERE, amount of reputation required = ??

### 5. Obtaining an API key for increased access

Having an API key will enable your (UTC-based) daily quota to go up from 300 to 10000, preventing potential rate limit issues and encouraging further experimentation.

1. Sign up for an account with stackapps.com by clicking on the "Sign up" button at the top right of the page. You can then use your Google or other email account associated with a Facebook (Meta) account.
2. Once signed into stackapps.com click on "Register an application" seen on the right side of the page.
3. In the "Register your application" page enter a name for the app and brief description (e.g. data science lab for education purposes). Agree to the terms and conditions and click on the "Register application" button.
4. Click the "Generate new API Key" button and paste the key (a string) into a text file (with no other characters) called "stackexchangekey.txt".

In [ ]:
# read API key from stackexchangekey.txt
try:
    with open(r"stackexchangekey.txt", "r") as f:
        API_KEY = f.read().strip()
    print("API key loaded successfully.")

except FileNotFoundError:
    print("stackexchangekey.txt not found.")

In [ ]:
# Create a new API instance now with API_KEY
SITE = StackAPI('stackoverflow', key=API_KEY)

### 6. Fetching a question by id

Let us look up a commonly asked question in Python - what the different types of brackets (round, square, curly) represent. We can use the search feature.

In [ ]:
search_results = SITE.fetch('search', intitle='meanings of brackets', tagged='python')

Looking at `search_results['items']` we find that the desired question_id is 30700603.

In [ ]:
search_results

In [ ]:
search_results['items']

In [ ]:
bracketid = search_results['items'][0]['question_id']
bracketid

Now we can fetch the question corresponding to this id.

In [ ]:
question = SITE.fetch(
    'questions/{ids}',
    ids=[bracketid],
    filter='withbody'
)

In [ ]:
print('Title', question['items'][0]['title'])
print('Link:', question['items'][0]['link'])
print('Score:', question['items'][0]['score'])
print('Tags:', question['items'][0]['tags'])

In [ ]:
print(question['items'][0]['body'])

To display this as Markdown:

In [ ]:
display(HTML(question['items'][0]['body']))

**Exercise**: Fetch the top-rated answer to this question displaying the answer as Markdown using the display(HTML()) function. Also find its score (number of upvotes - number of downvotes). Hint: use 'questions/{ids}/answers'.

In [ ]:
# ANSWER

### 7. Exploring tags

In [ ]:
popular_tags = SITE.fetch('tags', sort='popular', order='desc')

**Exercise**: Explore this object and print the 20 tags with the highest count in descending order. (Note: The sorted function (applied to a dictionary) may be helpful here. Expect javascript and python at or near the top of the list.)

In [ ]:
# ANSWER


In [ ]:
# ANSWER


### 8. Early adopters of the platform

**Exercise**: On what date did Stackoverflow (not datascience.stackexchange) first admit users? How many users signed up on this day? (use pd.to_datetime on the creation_date without specifying a time zone)

In [ ]:
# ANSWER


### 9. Creating a Dataframe

Observe how the following code creates a Pandas dataframe with the most recent questions.

In [ ]:
questions = SITE.fetch(
    'questions',
    pagesize=50,
    order='desc',
    sort='creation',
    filter='default'  # include tags, owner, score, etc.
)

The Dataframe function can take in a list of dictionaries, each representing a row of data:

In [ ]:
data = questions['items']

df_questions = pd.DataFrame([{
    'question_id': q['question_id'],
    'title': q['title'],
    'tags': q['tags'],
    'creation_date': pd.to_datetime(q['creation_date'], unit='s'),
    'score': q['score'],
    'view_count': q['view_count'],
    'answer_count': q['answer_count'],
    'is_answered': q['is_answered'],
    'owner_user_id': q['owner']['user_id'] if 'user_id' in q['owner'] else None
} for q in data])

df_questions.head()

**Exercise**: In a similar manner create a dataframe showing the 10 users with highest reputation.

Column names:

- Display_name
- User_id
- Number_of_gold_badges
- Reputation

Hint: fetch 'users' sorting by reputation in descending order, then view the object retrieved and extract the fields listed above.

In [ ]:
# ANSWER


### 10. A visualisation of popular tags

The following extracts the top 10 tags in the `datascience` network by popularity.

In [ ]:
SITE = StackAPI('datascience', key=API_KEY)

data = SITE.fetch(
    'tags',
    pagesize=10,
    order='desc',
    sort='popular'
)

**Exercise**: inspect the `data` dictionary and then create a variable tags_counts that is a list of tuples containing the name and count of each item

In [ ]:
# ANSWER - expect tags_counts to have the form
# [(tag1, count1), (tag2, count2), ...] where each tag is a string and each count is a number



The following cell sorts and then plots the counts.

In [ ]:
tags_counts.sort(key=lambda x: x[1], reverse=True)
tags, counts = zip(*tags_counts) # the * unpacks the list so that each tuple is an argument in the zip function

# Plot
plt.figure(figsize=(10,6))
bars = plt.bar(tags[:10], counts[:10], color='lightgreen')
plt.title("Top 10 Tags by Total Questions on Data Science")
plt.ylabel("Number of Questions")
plt.xticks(rotation=45, ha='right')

for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height()+100, str(count), ha='center')

plt.tight_layout() # helps fix spacing automatically so that labels, titles, and axes don’t overlap or get cut off
plt.show()


### Bonus: Further exploration

Explore one other Stackexchange site and either:

- show popular questions with highest scoring answers as a dataframe

or

- show highest scoring question ids as a bar chart

In [ ]:
# ANSWER


### Bonus: Authorisation Code grant with PKCE

If you wish to obtain access to your personal information, you need to sign up for an access token. This prevents others from seeing your details without permission.

Running the following cell will cause a StackAPI`Error` since an access token is required.

In [ ]:
me = SITE.fetch('me')

print(me['items'][0]['display_name'])
print(me['items'][0]['reputation'])

Obtaining an access token in Stackexchange is complex (see https://stackapps.com/help/api-authentication) and requires an Authorization Code grant with PKCE (Proof Key for Code Exchange). The following code automates this for you.

A random string called a code verifier is generated and then a SHA-256 hashed value of this called a code challenge is produced. This string will be used to generate a URL which when accessed will give back an authorization code. This code will be sent via a second API request and exchanged for a token response. This final token response string will be used to access your personal credentials.

In the cell below you will need to enter your client id. This can be found by navigating to your app in https://stackapps.com/applications/ and noting the five digits at the end of the URL.

In [ ]:
import os
import base64
import hashlib
import secrets
import requests
import webbrowser
from urllib.parse import urlencode

CLIENT_ID = "12345" # CHANGE to your five-digit string
REDIRECT_URI = "https://stackexchange.com/oauth/login_success"
AUTH_URL = "https://stackoverflow.com/oauth"
TOKEN_URL = "https://stackoverflow.com/oauth/access_token"

# 1: Generate PKCE code_verifier
code_verifier = base64.urlsafe_b64encode(
    secrets.token_bytes(32)
).rstrip(b'=').decode('utf-8')

# 2: Generate code_challenge
code_challenge = base64.urlsafe_b64encode(
    hashlib.sha256(code_verifier.encode()).digest()
).rstrip(b'=').decode('utf-8')

# 3: Build authorization URL
params = {
    "client_id": CLIENT_ID,
    "response_type": "code",
    "redirect_uri": REDIRECT_URI,
    "scope": "no_expiry",
    "code_challenge": code_challenge,
    "code_challenge_method": "S256",
}

auth_url = f"{AUTH_URL}?{urlencode(params)}"

print("\nOpen this URL in your browser:\n")
print(auth_url)
webbrowser.open(auth_url)

# 4: Paste authorization code
code = input("\nPaste the 'code' parameter from the end of the new redirect URL here:\n> ").strip()

# 5: Exchange code for access token
token_response = requests.post(TOKEN_URL, data={
    "client_id": CLIENT_ID,
    "code": code,
    "redirect_uri": REDIRECT_URI,
    "grant_type": "authorization_code",
    "code_verifier": code_verifier,
})

print("\nRaw token response:\n")
print(token_response.text)



If a raw token response is given the following should work:

In [ ]:
# StackExchange returns a form-encoded response
token_data = dict(
    item.split("=") for item in token_response.text.split("&")
)

ACCESS_TOKEN = token_data.get("access_token")

if not ACCESS_TOKEN:
    print("\nFailed to get access token")
    exit()

print("\nAccess token obtained!")

With this access token we can now access information corresponding to your account.

In [ ]:
SITE = StackAPI('stackoverflow', key=API_KEY, access_token=ACCESS_TOKEN)

me_response = SITE.fetch('me')

print("\n/me response:\n")
print(me_response)

This can be used to access various details about yourself:

In [ ]:
SITE = StackAPI('stackoverflow', key=API_KEY, access_token=ACCESS_TOKEN)

me = SITE.fetch('me')

print(me['items'][0]['display_name'])
print(me['items'][0]['reputation'])

**Optional activity**: favourite then unfavourite a post by question id by using https://api.stackexchange.com/2.3/questions/{question_id}/favorite and https://api.stackexchange.com/2.3/questions/{question_id}/unfavorite (this is only possible if you have already created a post on the stackexchange network with your account).

Note: this can be done with the requests library, not StackAPI.

In [ ]:
# ANSWER
# favourite a post:


In [ ]:
# ANSWER
# unfavourite a post:


### Conclusion

This lab introduced you to working with the Stack Exchange API using the stackapi Python library. After creating a site object you are able to interact with a specific community and practise retrieving real data such as questions by ID, exploring popular tags, and identifying early adopters of the platform. Signing up for an API provides further access. Also the retrieved JSON data can be converted into a structured pandas DataFrame or graphic,  reinforcing skills in API interaction, data wrangling, and data visualisation.

This API can be utilised in many ways:

- finding influential or viral questions
- studying the quality or speed of responses
- user reputation analysis
- comparing multiple Stack Exchange sites



---



---



> > > > > > > > > © 2026 Institute of Data


---



---



